# Can we predict 30-day readmission?
Data: `data.csv` (one row per hospital stay), dictionary in `columns.csv`. Target: `readmit_30d`.

Key design point: the 11,868 stays belong to only 1,500 people (~8 stays each). Stays of the same person are not independent, so **all validation is grouped by `person_key`**; a random row split would let the model see the same person in train and test and overstate performance.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',150)
df=pd.read_csv('data.csv'); cols=pd.read_csv('columns.csv')
y=df.readmit_30d; groups=df.person_key
print(df.shape, 'people:',groups.nunique(), '| prevalence: %.3f'%y.mean())
df.describe().T

## Data quality

In [ ]:
print('missing:',df.isna().sum().sum(),'| duplicate rows:',df.duplicated().sum())
g=df.groupby('person_key').visit_number
print('persons with duplicate visit_number:',(g.nunique()!=g.size).sum() if False else (g.nunique()!=g.size()).sum(),'| persons with gaps:',((g.max()-g.min()+1)!=g.size()).sum())
df.hist(figsize=(12,8),bins=30); plt.tight_layout()

## Univariate signal
Readmission rate by decile of each predictor.

In [ ]:
num=['visit_number','birth_year','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,c in zip(ax.ravel(),num):
    b=pd.qcut(df[c],10,duplicates='drop')
    r=y.groupby(b,observed=True).mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(y.mean(),ls='--',c='grey'); a.set_title(c); a.set_ylim(0,.5)
plt.tight_layout()
print(y.groupby(df.sex).mean())
print(df[num].corrwith(y).round(3))

## Models
Grouped 5-fold CV (stratified, no person in both train and test). Baselines: prevalence-only and logistic regression; then gradient boosting. Metrics: ROC AUC, average precision (chance = prevalence), Brier score.

In [ ]:
X=df.drop(columns=['readmit_30d','person_key']).copy()
X['sex']=(X.sex=='M').astype(int)
cv=StratifiedGroupKFold(5,shuffle=True,random_state=0)
models={
 'prevalence':DummyClassifier(strategy='prior'),
 'logreg':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'hist_gb':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,min_samples_leaf=50,random_state=0),
}
oof={};rows=[]
for n,m in models.items():
    p=cross_val_predict(m,X,y,groups=groups,cv=cv,method='predict_proba')[:,1]; oof[n]=p
    per=[roc_auc_score(y[t],p[t]) for _,t in cv.split(X,y,groups)]
    rows.append(dict(model=n,auc=roc_auc_score(y,p),auc_fold_sd=np.std(per),ap=average_precision_score(y,p),brier=brier_score_loss(y,p)))
res=pd.DataFrame(rows).set_index('model').round(4); res

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
for n in ['logreg','hist_gb']:
    f,t,_=roc_curve(y,oof[n]); ax[0].plot(f,t,label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend(); ax[0].set_title('ROC (grouped CV)')
for n in ['logreg','hist_gb']:
    fp,mp=calibration_curve(y,oof[n],n_bins=10,strategy='quantile'); ax[1].plot(mp,fp,'o-',label=n)
ax[1].plot([0,.5],[0,.5],'k--'); ax[1].set_xlabel('predicted'); ax[1].set_ylabel('observed'); ax[1].set_title('Calibration'); ax[1].legend()
plt.tight_layout()

### Uncertainty: person-level (cluster) bootstrap of the out-of-fold AUC

In [ ]:
rng=np.random.default_rng(0)
ids=groups.unique(); idx=df.groupby('person_key').indices
for n in ['logreg','hist_gb']:
    p=oof[n]; aucs=[]
    for _ in range(500):
        s=np.concatenate([idx[i] for i in rng.choice(ids,len(ids))])
        aucs.append(roc_auc_score(y.values[s],p[s]))
    print('%s AUC 95%% CI: %.3f-%.3f'%((n,)+tuple(np.percentile(aucs,[2.5,97.5]))))

## What drives the prediction?

In [ ]:
lr=models['logreg'].fit(X,y)
coef=pd.Series(lr[-1].coef_[0],index=X.columns).sort_values()
print('Standardised logistic coefficients (log-odds per 1 SD):'); print(coef.round(3))
imp=np.zeros(X.shape[1])
for tr,te in cv.split(X,y,groups):
    m=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)).fit(X.iloc[tr],y.iloc[tr])
    imp+=permutation_importance(m,X.iloc[te],y.iloc[te],scoring='roc_auc',n_repeats=5,random_state=0).importances_mean/5
pd.Series(imp,index=X.columns).sort_values().plot.barh(title='Held-out permutation importance (AUC drop, logreg)');

## Utility: does a risk ranking concentrate readmissions?

In [ ]:
d=pd.DataFrame({'y':y,'p':oof['logreg']})
d['decile']=pd.qcut(d.p.rank(method='first'),10,labels=range(1,11))
t=d.groupby('decile',observed=True).agg(n=('y','size'),readmit_rate=('y','mean'),mean_pred=('p','mean')).round(3)
t['lift']=(t.readmit_rate/y.mean()).round(2); display(t)
top=d.nlargest(int(len(d)*.2),'p')
print('Top 20%% by risk: precision %.3f (base %.3f), captures %.1f%% of readmissions'%(top.y.mean(),y.mean(),100*top.y.sum()/y.sum()))

## Conclusions
- **Yes, but only modestly.** Out-of-fold (person-grouped) ROC AUC is ~0.65 for logistic regression (95% CI 0.63–0.66) vs 0.50 for the prevalence baseline; average precision 0.32 vs 0.20 chance. This is real signal, but weak for individual-level decisions.
- **Drivers:** longer stay, more discharge medications and higher `lab_a` raise risk (≈ +0.3 log-odds per SD each). Other columns (sex, age, BMI, postcode, `lab_b`, visit number) show essentially no effect.
- **Simple beats complex:** gradient boosting (AUC ~0.64) is no better than logistic regression, suggesting little non-linear/interaction structure in these features.
- **Calibration is good**; the top risk decile has ~40% readmission (2x lift) vs ~9% in the bottom decile. Flagging the top 20% catches ~34% of readmissions at 34% precision.
- **Caveats:** repeated stays per person were handled with grouped CV, but no prior-visit history features (e.g. previous readmission) were engineered; the data have no timestamps, so temporal validation wasn't possible. Predictors are all known at/near discharge, so no obvious target leakage, but `n_medications` is "at discharge" and should be confirmed available at prediction time.
- Note: cell outputs were generated by executing the cells with a plain Python runner (the Jupyter kernel could not start in the sandbox); re-run in Jupyter to refresh.